In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

In [3]:
def _match_from_stem(stem: str) -> str:
    """Zwraca nazwę meczu na podstawie nazwy pliku (stem)."""
    if stem.startswith("skip_"):
        parts = stem.split("_")
        return "_".join(parts[:2]) if len(parts) >= 2 else stem
    if stem[:2].isdigit():
        return "m" + stem[:2]
    if stem.startswith("pic_"):
        return "m_pic"
    return "m_" + stem.split("_")[0] if "_" in stem else "m_other"


def build_shot_labels_df(pic_dir: Path = Path("../pics"), skip_dir: Path = Path("../pics/skip")):

    skip_img_names = [pic.stem for pic in skip_dir.glob("*.png")]
    zero_labels = np.zeros(len(skip_img_names), dtype=np.uint8)

    img_names = [pic.stem for pic in pic_dir.glob("*.png")]
    one_labels = np.ones(len(img_names), dtype=np.uint8)

    img_df = pd.DataFrame({"img_name": img_names, "label": one_labels})
    skip_df = pd.DataFrame({"img_name": skip_img_names, "label": zero_labels})

    df = pd.concat([img_df, skip_df]).sample(frac=1).reset_index(drop=True)
    df["match"] = df["img_name"].map(_match_from_stem)

    return df


In [4]:
skip_dir = Path("../pics/skip")
pic_dir = Path("../pics")

In [5]:
df = build_shot_labels_df()

In [6]:
df

,img_name,label,match
0,pic_09_23_01,1,m_pic
1,pic_05_37_01,1,m_pic
2,03_024,1,m03
3,pic_02_78_01,1,m_pic
4,skip_m07_0526,0,skip_m07
...,...,...,...
1805,skip_m07_0570,0,skip_m07
1806,skip_m05_0424,0,skip_m05
1807,skip_m01_0009,0,skip_m01
1808,02_019,1,m02


In [5]:
df['label'].value_counts(normalize=True)

label
1    0.518785
0    0.481215
Name: proportion, dtype: float64

In [6]:
sorted(df['match'].unique())

['m01',
 'm02',
 'm03',
 'm04',
 'm05',
 'm06',
 'm_pic',
 'skip_m01',
 'skip_m02',
 'skip_m03',
 'skip_m04',
 'skip_m05',
 'skip_m06',
 'skip_m07',
 'skip_m08',
 'skip_m09',
 'skip_m10']

In [7]:
df.to_csv("shot_labels.csv", index=False)